In [ ]:
# ==========================================
# 1. 載入必要套件
# ==========================================
import os
import chardet
import numpy as np
import pandas as pd
from scipy.stats import norm
from google.colab import drive

# ==========================================
# 2. 自定義設定區 (請依據實際逐筆資料欄位修改)
# ==========================================
FOLDER_PATH = '/content/drive/MyDrive/金融資料探勘'
INDEX_FILE = 'Path_教學_0409.csv'

# 逐筆交易檔案(OptionsDaily_*.csv)內的欄位名稱對應
COL_CONTRACT = '到期月份(週別)'  # 契約月份
COL_CP = '買賣權別'            # 買賣權 (依據你的檔案，通常是 '買賣權別')
COL_STRIKE = '履約價格'          # 履約價
COL_PRICE = '成交價格'          # 市場成交價格
CALL_FLAG = 'C'                # 判斷為買權的字串標記 (依據你的檔案為 'C')

# ==========================================
# 3. 雲端整合：掛載 Google Drive
# ==========================================
print("正在掛載 Google Drive...")
drive.mount('/content/drive')
base_path = FOLDER_PATH
index_file_path = os.path.join(base_path, INDEX_FILE)

# ==========================================
# 4. 定義數理模型：BS 模型與二分法推導 IV
# ==========================================
def bs_call_price(S, K, T, r, sigma):
    """計算 Black-Scholes Call Option 理論價格"""
    if T <= 0 or sigma <= 0:
        return max(S - K, 0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def implied_volatility_bisection(S, K, T, r, market_price, tol=1e-5, max_iter=100):
    """使用二分法 (Bisection Method) 反推隱含波動率 (IV)"""
    if market_price < max(S - K, 0):
        return np.nan

    low_vol, high_vol = 1e-5, 5.0
    for _ in range(max_iter):
        mid_vol = (low_vol + high_vol) / 2
        price_mid = bs_call_price(S, K, T, r, mid_vol)

        if abs(price_mid - market_price) < tol:
            return mid_vol

        if price_mid > market_price:
            high_vol = mid_vol
        else:
            low_vol = mid_vol

    return (low_vol + high_vol) / 2

# ==========================================
# 5. 資料處理：編碼偵測函數
# ==========================================
def detect_encoding(file_path):
    """自動偵測檔案編碼"""
    with open(file_path, 'rb') as f:
        raw_data = f.read(10000)
    result = chardet.detect(raw_data)
    encoding = result['encoding']
    return encoding if encoding else 'utf-8'

# ==========================================
# 6. 主程式：讀取索引檔並處理逐筆資料
# ==========================================
if not os.path.exists(index_file_path):
    print(f"找不到索引檔：{index_file_path}，請確認路徑。")
else:
    df_index = pd.read_csv(index_file_path)
    print(f"成功讀取索引檔，共 {len(df_index)} 筆設定資料。\n")

    final_results = []

    for index, row in df_index.iterrows():
        date = row['Date']
        file_name = row['File']
        S0 = row['S0']
        T = row['Maturity'] / 365.0
        target_contract = str(row['Contract'])
        Rf = row['Rf']

        daily_file_path = os.path.join(base_path, file_name)

        if not os.path.exists(daily_file_path):
            print(f"警告：找不到逐筆資料檔案 {file_name}，已略過。")
            continue

        file_encoding = detect_encoding(daily_file_path)
        try:
            df_daily = pd.read_csv(daily_file_path, encoding=file_encoding, on_bad_lines='skip')
        except Exception as e:
            print(f"讀取 {file_name} 失敗: {e}")
            continue

        # ==========================================
        # 🌟 更新的資料清洗區塊 🌟
        # ==========================================
        # 1. 刪除所有表頭（包含第一欄）前後的空格
        df_daily.columns = df_daily.columns.str.strip()

        # 2. 刪除包含 '---' 的那一列（通常在資料的第一列）
        # 檢查第一欄（成交日期）是否包含 '-'，若有則排除該列，藉此把分隔線刪掉
        df_daily = df_daily[~df_daily.iloc[:, 0].astype(str).str.contains("-", na=False)]
        # ==========================================

        required_cols = [COL_CONTRACT, COL_CP, COL_STRIKE, COL_PRICE]
        if not all(col in df_daily.columns for col in required_cols):
            print(f"警告：{file_name} 缺少必要欄位，請檢查自定義設定區。")
            continue

        df_daily[COL_CONTRACT] = df_daily[COL_CONTRACT].astype(str).str.strip()
        df_call = df_daily[
            (df_daily[COL_CONTRACT] == target_contract) &
            (df_daily[COL_CP].astype(str).str.contains(CALL_FLAG, case=False, na=False))
        ].copy()

        df_call = df_call.dropna(subset=[COL_STRIKE, COL_PRICE])

        if df_call.empty:
            print(f"日期 {date} ({file_name}) 找不到符合條件的 Call 交易紀錄。")
            continue

        print(f"正在處理 {date} 的資料 ({file_name})，共 {len(df_call)} 筆 Call 紀錄...")

        df_call['Implied_Volatility'] = df_call.apply(
            lambda x: implied_volatility_bisection(
                S=S0,
                K=float(x[COL_STRIKE]),
                T=T,
                r=Rf,
                market_price=float(x[COL_PRICE])
            ), axis=1
        )

        df_call['Date'] = date
        df_call['S0'] = S0
        df_call['Rf'] = Rf
        df_call['Maturity_Days'] = row['Maturity']

        final_results.append(df_call)

    # ==========================================
    # 7. 彙整結果與輸出
    # ==========================================
    if final_results:
        df_final = pd.concat(final_results, ignore_index=True)
        print("\n資料處理與 IV 計算完成！以下為前 5 筆資料預覽：")
        display(df_final.head())

        output_path = os.path.join(base_path, 'Processed_Options_IV.csv')
        df_final.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"✅ 最終結果已儲存至：{output_path}")
    else:
        print("沒有處理出任何結果，請檢查資料或過濾條件。")

正在掛載 Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
成功讀取索引檔，共 5 筆設定資料。



/tmp/ipykernel_2236/651087670.py:101: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_daily = pd.read_csv(daily_file_path, encoding=file_encoding, on_bad_lines='skip')


正在處理 2020/1/2 的資料 (OptionsDaily_2020_01_02.csv)，共 28332 筆 Call 紀錄...


/tmp/ipykernel_2236/651087670.py:101: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_daily = pd.read_csv(daily_file_path, encoding=file_encoding, on_bad_lines='skip')


正在處理 2020/1/3 的資料 (OptionsDaily_2020_01_03.csv)，共 47961 筆 Call 紀錄...


...

正在處理 2020/1/7 的資料 (OptionsDaily_2020_01_07.csv)，共 55180 筆 Call 紀錄...


/tmp/ipykernel_2236/651087670.py:101: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_daily = pd.read_csv(daily_file_path, encoding=file_encoding, on_bad_lines='skip')


正在處理 2020/1/8 的資料 (OptionsDaily_2020_01_08.csv)，共 71598 筆 Call 紀錄...

資料處理與 IV 計算完成！以下為前 5 筆資料預覽：


,成交日期,商品代號,履約價格,到期月份(週別),買賣權別,成交時間,成交價格,成交數量(B or S),開盤集合競價,Implied_Volatility,Date,S0,Rf,Maturity_Days
0,20200102,CBO,24.0,202001,C,104303.0,0.14,2.0,,NaN,2020/1/2,12100.48,0.0109,13
1,20200102,CBO,24.0,202001,C,104303.0,0.14,2.0,,NaN,2020/1/2,12100.48,0.0109,13
2,20200102,CDO,330.0,202001,C,90414.0,7.00,1.0,,NaN,2020/1/2,12100.48,0.0109,13
3,20200102,CDO,330.0,202001,C,90414.0,7.00,1.0,,NaN,2020/1/2,12100.48,0.0109,13
4,20200102,CDO,330.0,202001,C,100532.0,8.10,1.0,,NaN,2020/1/2,12100.48,0.0109,13


✅ 最終結果已儲存至：/content/drive/MyDrive/金融資料探勘/Processed_Options_IV.csv


In [ ]:
# ==========================================
# 1. 載入必要套件
# ==========================================
import os
import chardet
import numpy as np
import pandas as pd
from scipy.stats import norm
from google.colab import drive

# ==========================================
# 2. 自定義設定區 (請依據實際逐筆資料欄位修改)
# ==========================================
FOLDER_PATH = '/content/drive/MyDrive/金融資料探勘'
INDEX_FILE = 'Path_教學_0409.csv'

# 逐筆交易檔案內的欄位名稱對應
COL_CONTRACT = '到期月份(週別)'
COL_CP = '買賣權別'
COL_STRIKE = '履約價格'
COL_PRICE = '成交價格'
CALL_FLAG = 'C'

# ==========================================
# 3. 雲端整合：掛載 Google Drive
# ==========================================
print("正在掛載 Google Drive...")
drive.mount('/content/drive')
base_path = FOLDER_PATH
index_file_path = os.path.join(base_path, INDEX_FILE)

# ==========================================
# 4. 定義數理模型：BS 模型與二分法推導 IV
# ==========================================
def bs_call_price(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0:
        return max(S - K, 0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def implied_volatility_bisection(S, K, T, r, market_price, tol=1e-5, max_iter=100):
    if market_price < max(S - K, 0):
        return np.nan

    low_vol, high_vol = 1e-5, 5.0
    for _ in range(max_iter):
        mid_vol = (low_vol + high_vol) / 2
        price_mid = bs_call_price(S, K, T, r, mid_vol)

        if abs(price_mid - market_price) < tol:
            return mid_vol

        if price_mid > market_price:
            high_vol = mid_vol
        else:
            low_vol = mid_vol

    return (low_vol + high_vol) / 2

# ==========================================
# 5. 資料處理：編碼偵測函數
# ==========================================
def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        raw_data = f.read(10000)
    result = chardet.detect(raw_data)
    encoding = result['encoding']
    return encoding if encoding else 'utf-8'

# ==========================================
# 6. 主程式：讀取索引檔並處理逐筆資料
# ==========================================
if not os.path.exists(index_file_path):
    print(f"找不到索引檔：{index_file_path}，請確認路徑。")
else:
    df_index = pd.read_csv(index_file_path)
    print(f"成功讀取索引檔，共 {len(df_index)} 筆設定資料。\n")

    final_results = []

    for index, row in df_index.iterrows():
        # 從索引檔抓取你需要的所有欄位
        date = row['Date']
        file_name = row['File']
        S0 = row['S0']
        T = row['Maturity'] / 365.0
        target_contract = str(row['Contract'])
        contract_expiry = row['ContractExpiryDate']
        Rf = row['Rf']

        daily_file_path = os.path.join(base_path, file_name)

        if not os.path.exists(daily_file_path):
            print(f"警告：找不到逐筆資料檔案 {file_name}，已略過。")
            continue

        file_encoding = detect_encoding(daily_file_path)
        try:
            df_daily = pd.read_csv(daily_file_path, encoding=file_encoding, on_bad_lines='skip')
        except Exception as e:
            print(f"讀取 {file_name} 失敗: {e}")
            continue

        # 資料清洗區塊
        df_daily.columns = df_daily.columns.str.strip()
        df_daily = df_daily[~df_daily.iloc[:, 0].astype(str).str.contains("-", na=False)]

        required_cols = [COL_CONTRACT, COL_CP, COL_STRIKE, COL_PRICE]
        if not all(col in df_daily.columns for col in required_cols):
            print(f"警告：{file_name} 缺少必要欄位，請檢查自定義設定區。")
            continue

        df_daily[COL_CONTRACT] = df_daily[COL_CONTRACT].astype(str).str.strip()
        df_call = df_daily[
            (df_daily[COL_CONTRACT] == target_contract) &
            (df_daily[COL_CP].astype(str).str.contains(CALL_FLAG, case=False, na=False))
        ].copy()

        df_call = df_call.dropna(subset=[COL_STRIKE, COL_PRICE])

        if df_call.empty:
            print(f"日期 {date} ({file_name}) 找不到符合條件的 Call 交易紀錄。")
            continue

        print(f"正在處理 {date} 的資料 ({file_name})，共 {len(df_call)} 筆 Call 紀錄...")

        df_call['Implied_Volatility'] = df_call.apply(
            lambda x: implied_volatility_bisection(
                S=S0,
                K=float(x[COL_STRIKE]),
                T=T,
                r=Rf,
                market_price=float(x[COL_PRICE])
            ), axis=1
        )

        # 🌟 寫入你指定的 6 個欄位 🌟
        df_call['Date'] = date
        df_call['File'] = file_name
        df_call['S0'] = S0
        df_call['Contract'] = target_contract
        df_call['ContractExpiryDate'] = contract_expiry
        df_call['Rf'] = Rf

        final_results.append(df_call)

    # ==========================================
    # 7. 彙整結果、過濾指定欄位與輸出
    # ==========================================
    if final_results:
        df_final = pd.concat(final_results, ignore_index=True)

        # 🌟 篩選最終輸出的欄位 🌟
        # 保留你要求的 6 個欄位，加上必要的交易資料(履約價、成交價)與算出的 IV
        keep_columns = [
            'Date', 'File', 'S0', 'Contract', 'ContractExpiryDate', 'Rf',
            COL_STRIKE, COL_PRICE, 'Implied_Volatility'
        ]
        df_final = df_final[keep_columns]

        print("\n資料處理與 IV 計算完成！以下為前 5 筆資料預覽：")
        display(df_final.head())

        output_path = os.path.join(base_path, 'Processed_Options_IV_Filtered.csv')
        df_final.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"✅ 最終結果已儲存至：{output_path}")
    else:
        print("沒有處理出任何結果，請檢查資料或過濾條件。")

正在掛載 Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
成功讀取索引檔，共 5 筆設定資料。



/tmp/ipykernel_2236/2418576368.py:100: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_daily = pd.read_csv(daily_file_path, encoding=file_encoding, on_bad_lines='skip')


正在處理 2020/1/2 的資料 (OptionsDaily_2020_01_02.csv)，共 28332 筆 Call 紀錄...


/tmp/ipykernel_2236/2418576368.py:100: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_daily = pd.read_csv(daily_file_path, encoding=file_encoding, on_bad_lines='skip')


正在處理 2020/1/3 的資料 (OptionsDaily_2020_01_03.csv)，共 47961 筆 Call 紀錄...


...

正在處理 2020/1/7 的資料 (OptionsDaily_2020_01_07.csv)，共 55180 筆 Call 紀錄...


/tmp/ipykernel_2236/2418576368.py:100: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_daily = pd.read_csv(daily_file_path, encoding=file_encoding, on_bad_lines='skip')


正在處理 2020/1/8 的資料 (OptionsDaily_2020_01_08.csv)，共 71598 筆 Call 紀錄...

資料處理與 IV 計算完成！以下為前 5 筆資料預覽：


,Date,File,S0,Contract,ContractExpiryDate,Rf,履約價格,成交價格,Implied_Volatility
0,2020/1/2,OptionsDaily_2020_01_02.csv,12100.48,202001,2020/1/15,0.0109,24.0,0.14,NaN
1,2020/1/2,OptionsDaily_2020_01_02.csv,12100.48,202001,2020/1/15,0.0109,24.0,0.14,NaN
2,2020/1/2,OptionsDaily_2020_01_02.csv,12100.48,202001,2020/1/15,0.0109,330.0,7.00,NaN
3,2020/1/2,OptionsDaily_2020_01_02.csv,12100.48,202001,2020/1/15,0.0109,330.0,7.00,NaN
4,2020/1/2,OptionsDaily_2020_01_02.csv,12100.48,202001,2020/1/15,0.0109,330.0,8.10,NaN


✅ 最終結果已儲存至：/content/drive/MyDrive/金融資料探勘/Processed_Options_IV_Filtered.csv


In [ ]:
# ==========================================
# 1. 載入必要套件
# ==========================================
import os
import chardet
import numpy as np
import pandas as pd
from scipy.stats import norm
from google.colab import drive

# ==========================================
# 2. 自定義設定區
# ==========================================
FOLDER_PATH = '/content/drive/MyDrive/金融資料探勘'
INDEX_FILE = 'Path_教學_0409.csv'

COL_CONTRACT = '到期月份(週別)'
COL_CP = '買賣權別'
COL_STRIKE = '履約價格'
COL_PRICE = '成交價格'
COL_VOLUME = '成交數量(B or S)'  # 新增：成交數量欄位
CALL_FLAG = 'C'

# ==========================================
# 3. 掛載 Google Drive
# ==========================================
print("正在掛載 Google Drive...")
drive.mount('/content/drive')
base_path = FOLDER_PATH
index_file_path = os.path.join(base_path, INDEX_FILE)

# ==========================================
# 4. BS 模型與二分法
# ==========================================
def bs_call_price(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0:
        return max(S - K, 0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def implied_volatility_bisection(S, K, T, r, market_price, tol=1e-5, max_iter=100):
    if market_price < max(S - K, 0):
        return np.nan

    low_vol, high_vol = 1e-5, 5.0
    for _ in range(max_iter):
        mid_vol = (low_vol + high_vol) / 2
        price_mid = bs_call_price(S, K, T, r, mid_vol)

        if abs(price_mid - market_price) < tol:
            return mid_vol

        if price_mid > market_price:
            high_vol = mid_vol
        else:
            low_vol = mid_vol

    return (low_vol + high_vol) / 2

# ==========================================
# 5. 編碼偵測
# ==========================================
def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        raw_data = f.read(10000)
    result = chardet.detect(raw_data)
    encoding = result['encoding']
    return encoding if encoding else 'utf-8'

# ==========================================
# 6. 主程式：讀取並極速運算
# ==========================================
if not os.path.exists(index_file_path):
    print(f"找不到索引檔：{index_file_path}")
else:
    df_index = pd.read_csv(index_file_path)
    print(f"成功讀取索引檔，共 {len(df_index)} 筆設定。\n")

    final_results = []

    for index, row in df_index.iterrows():
        date = row['Date']
        file_name = row['File']
        S0 = row['S0']
        T = row['Maturity'] / 365.0
        target_contract = str(row['Contract'])
        contract_expiry = row['ContractExpiryDate']
        Rf = row['Rf']

        daily_file_path = os.path.join(base_path, file_name)
        if not os.path.exists(daily_file_path):
            continue

        file_encoding = detect_encoding(daily_file_path)
        try:
            df_daily = pd.read_csv(daily_file_path, encoding=file_encoding, on_bad_lines='skip')
        except Exception as e:
            print(f"讀取 {file_name} 失敗: {e}")
            continue

        # 1. 欄位清洗 (去空白與分隔線)
        df_daily.columns = df_daily.columns.str.strip()
        df_daily = df_daily[~df_daily.iloc[:, 0].astype(str).str.contains("-", na=False)]

        # 2. 確認所有必要欄位存在 (包含成交數量)
        required_cols = [COL_CONTRACT, COL_CP, COL_STRIKE, COL_PRICE, COL_VOLUME]
        if not all(col in df_daily.columns for col in required_cols):
            print(f"[{date}] 警告：缺少必要欄位，已略過。")
            continue

        # 3. 成交數量轉型與過濾
        # 強制轉型為浮點數(去除不可見字元)，再過濾 > 30 的資料
        df_daily[COL_VOLUME] = pd.to_numeric(df_daily[COL_VOLUME].astype(str).str.strip(), errors='coerce')
        df_daily[COL_CONTRACT] = df_daily[COL_CONTRACT].astype(str).str.strip()

        df_call = df_daily[
            (df_daily[COL_CONTRACT] == target_contract) &
            (df_daily[COL_CP].astype(str).str.contains(CALL_FLAG, case=False, na=False)) &
            (df_daily[COL_VOLUME] > 30)  # 🌟 新增：交易量超過 30 條件 🌟
        ].copy()

        # 4. 去除無效的價格資料
        df_call[COL_STRIKE] = pd.to_numeric(df_call[COL_STRIKE], errors='coerce')
        df_call[COL_PRICE] = pd.to_numeric(df_call[COL_PRICE], errors='coerce')
        df_call = df_call.dropna(subset=[COL_STRIKE, COL_PRICE])

        if df_call.empty:
            print(f"[{date}] 過濾後無符合條件 (Call 且 Volume>30) 的資料。")
            continue

        print(f"[{date}] 資料共 {len(df_call)} 筆 (已排除低交易量雜訊)，開始極速運算...")

        # ==========================================
        # 🚀 效能優化核心：建立不重複的(履約價,成交價)對照表
        # ==========================================
        unique_pairs = df_call[[COL_STRIKE, COL_PRICE]].drop_duplicates()
        unique_pairs['Implied_Volatility'] = unique_pairs.apply(
            lambda x: implied_volatility_bisection(
                S=S0, K=x[COL_STRIKE], T=T, r=Rf, market_price=x[COL_PRICE]
            ), axis=1
        )
        df_call = df_call.merge(unique_pairs, on=[COL_STRIKE, COL_PRICE], how='left')
        # ==========================================

        # 賦予指定欄位
        df_call['Date'] = date
        df_call['File'] = file_name
        df_call['S0'] = S0
        df_call['Contract'] = target_contract
        df_call['ContractExpiryDate'] = contract_expiry
        df_call['Rf'] = Rf

        final_results.append(df_call)

    # ==========================================
    # 7. 彙整、敘述統計與輸出
    # ==========================================
    if final_results:
        df_final = pd.concat(final_results, ignore_index=True)
        keep_columns = [
            'Date', 'File', 'S0', 'Contract', 'ContractExpiryDate', 'Rf',
            COL_STRIKE, COL_PRICE, 'Implied_Volatility'
        ]
        df_final = df_final[keep_columns]

        # 儲存明細資料
        output_path = os.path.join(base_path, 'Processed_Options_IV_Filtered.csv')
        df_final.to_csv(output_path, index=False, encoding='utf-8-sig')

        # 🌟 每日隱含波動率 (IV) 敘述統計 🌟
        print("\n" + "="*50)
        print("📊 每日隱含波動率 (IV) 敘述統計：")
        print("="*50)
        # 使用 groupby 和 describe 產出平均數、標準差、四分位數等
        iv_stats = df_final.groupby('Date')['Implied_Volatility'].describe()
        display(iv_stats)

        # 同步將統計結果存成另一份檔案
        stats_output_path = os.path.join(base_path, 'IV_Daily_Statistics.csv')
        iv_stats.to_csv(stats_output_path, encoding='utf-8-sig')

        print(f"\n✅ 明細資料儲存至：{output_path}")
        print(f"✅ 敘述統計儲存至：{stats_output_path}")
    else:
        print("沒有處理出任何結果，請檢查資料或過濾條件。")

正在掛載 Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
成功讀取索引檔，共 5 筆設定。



/tmp/ipykernel_2236/3627757451.py:97: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_daily = pd.read_csv(daily_file_path, encoding=file_encoding, on_bad_lines='skip')


[2020/1/2] 資料共 279 筆 (已排除低交易量雜訊)，開始極速運算...


/tmp/ipykernel_2236/3627757451.py:97: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_daily = pd.read_csv(daily_file_path, encoding=file_encoding, on_bad_lines='skip')


[2020/1/3] 資料共 527 筆 (已排除低交易量雜訊)，開始極速運算...


...

[2020/1/7] 資料共 1058 筆 (已排除低交易量雜訊)，開始極速運算...


/tmp/ipykernel_2236/3627757451.py:97: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_daily = pd.read_csv(daily_file_path, encoding=file_encoding, on_bad_lines='skip')


[2020/1/8] 資料共 1457 筆 (已排除低交易量雜訊)，開始極速運算...

📊 每日隱含波動率 (IV) 敘述統計：


,count,mean,std,min,25%,50%,75%,max
Date,,,,,,,,
2020/1/2,279.0,0.135700,0.012773,0.074820,0.130367,0.140028,0.142652,0.158027
2020/1/3,527.0,0.135398,0.021365,0.000010,0.123059,0.128726,0.149264,0.223518
2020/1/6,492.0,0.175931,0.014286,0.150146,0.166122,0.173865,0.182872,0.260070
2020/1/7,1046.0,0.166954,0.031225,0.127779,0.151532,0.164489,0.175583,0.567802
2020/1/8,1449.0,0.157439,0.020840,0.114404,0.143417,0.153261,0.168578,0.301928



✅ 明細資料儲存至：/content/drive/MyDrive/金融資料探勘/Processed_Options_IV_Filtered.csv
✅ 敘述統計儲存至：/content/drive/MyDrive/金融資料探勘/IV_Daily_Statistics.csv


In [ ]:
# ==========================================
# 1. 載入必要套件
# ==========================================
import os
import zipfile
import chardet
import numpy as np
import pandas as pd
from scipy.stats import norm
from google.colab import drive

# ==========================================
# 2. 自定義設定區
# ==========================================
FOLDER_PATH = '/content/drive/MyDrive/金融資料探勘'
INDEX_FILE = 'Index_411336018_2022.xlsx'

# 目標子資料夾名稱
DATA_SUBFOLDER = 'Option_2022'

COL_CONTRACT = '到期月份(週別)'
COL_CP = '買賣權別'
COL_STRIKE = '履約價格'
COL_PRICE = '成交價格'
COL_VOLUME = '成交數量(B or S)'
PUT_FLAG = 'P' # 🌟 修改1：篩選目標改為賣權 P 🌟

# ==========================================
# 3. 掛載 Google Drive 與路徑設定
# ==========================================
print("🔄 正在掛載 Google Drive...")
drive.mount('/content/drive')

base_path = FOLDER_PATH
index_file_path = os.path.join(base_path, INDEX_FILE)
data_path = os.path.join(base_path, DATA_SUBFOLDER)

if not os.path.exists(index_file_path):
    raise FileNotFoundError(f"❌ 找不到索引檔：{index_file_path}")

if not os.path.exists(data_path):
    raise FileNotFoundError(f"❌ 找不到子資料夾：{data_path}，請確認資料夾名稱。")

# ==========================================
# 4. 尋找與讀取壓縮檔專用函數
# ==========================================
def find_file_in_subfolders(base_dir, target_file):
    """在指定資料夾及其所有子資料夾中尋找檔案"""
    for root, dirs, files in os.walk(base_dir):
        if target_file in files:
            return os.path.join(root, target_file)
    return None

def read_csv_from_zip(zip_path, target_csv_name):
    """直接從 ZIP 壓縮檔中讀取 CSV 資料 (不落地解壓)"""
    with zipfile.ZipFile(zip_path, 'r') as z:
        if target_csv_name not in z.namelist():
            csv_files = [f for f in z.namelist() if f.endswith('.csv')]
            if not csv_files:
                raise FileNotFoundError("壓縮檔內找不到任何 CSV 檔案！")
            target_csv_name = csv_files[0]

        with z.open(target_csv_name) as f:
            raw_data = f.read(10000)
            encoding = chardet.detect(raw_data)['encoding'] or 'utf-8'

        with z.open(target_csv_name) as f:
            df = pd.read_csv(f, encoding=encoding, on_bad_lines='skip')

        return df

# ==========================================
# 5. 🌟 修改2：賣權的 BS 模型與二分法 🌟
# ==========================================
def bs_put_price(S, K, T, r, sigma):
    """計算 Black-Scholes 賣權價格"""
    if T <= 0 or sigma <= 0:
        return max(K - S, 0) # 賣權的內含價值是 K - S
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    # 賣權公式: K*e^(-rT)*N(-d2) - S*N(-d1)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def implied_volatility_bisection_put(S, K, T, r, market_price, tol=1e-5, max_iter=100):
    """使用二分法計算賣權的隱含波動率"""
    if market_price < max(K - S, 0): # 賣權內含價值防呆檢查
        return np.nan
    low_vol, high_vol = 1e-5, 5.0
    for _ in range(max_iter):
        mid_vol = (low_vol + high_vol) / 2
        price_mid = bs_put_price(S, K, T, r, mid_vol)
        if abs(price_mid - market_price) < tol:
            return mid_vol
        if price_mid > market_price:
            high_vol = mid_vol
        else:
            low_vol = mid_vol
    return (low_vol + high_vol) / 2

# ==========================================
# 6. 主程式：讀取索引檔並處理逐筆資料
# ==========================================
print("\n📊 正在讀取 Excel 索引檔...")
df_index = pd.read_excel(index_file_path)

if df_index['S0'].dtype == object:
    df_index['S0'] = df_index['S0'].astype(str).str.replace(',', '').astype(float)

print(f"✅ 成功讀取索引檔，共 {len(df_index)} 筆設定。\n")

final_results = []

for index, row in df_index.iterrows():
    date = str(row['Date']).split(' ')[0]
    target_csv_name = str(row['File']).strip()

    target_zip_name = target_csv_name.replace('.csv', '.zip')

    S0 = float(row['S0'])
    T = row['Maturity'] / 365.0
    target_contract = str(row['Contract']).split('.')[0]
    contract_expiry = str(row['ContractExpiryDate']).split(' ')[0]
    Rf = float(row['Rf'])

    daily_zip_path = find_file_in_subfolders(data_path, target_zip_name)

    if not daily_zip_path:
        print(f"⚠️ [{date}] 找不到壓縮檔：{target_zip_name}，已略過。")
        continue

    try:
        df_daily = read_csv_from_zip(daily_zip_path, target_csv_name)
    except Exception as e:
        print(f"❌ 讀取 {target_zip_name} 失敗: {e}")
        continue

    df_daily.columns = df_daily.columns.str.strip()
    df_daily = df_daily[~df_daily.iloc[:, 0].astype(str).str.contains("-", na=False)]

    required_cols = [COL_CONTRACT, COL_CP, COL_STRIKE, COL_PRICE, COL_VOLUME]
    if not all(col in df_daily.columns for col in required_cols):
        print(f"⚠️ [{date}] 警告：{target_csv_name} 缺少必要欄位，已略過。")
        continue

    df_daily[COL_VOLUME] = pd.to_numeric(df_daily[COL_VOLUME].astype(str).str.strip(), errors='coerce')
    df_daily[COL_CONTRACT] = df_daily[COL_CONTRACT].astype(str).str.strip()

    # 🌟 修改3：過濾出賣權資料 🌟
    df_put = df_daily[
        (df_daily[COL_CONTRACT] == target_contract) &
        (df_daily[COL_CP].astype(str).str.contains(PUT_FLAG, case=False, na=False)) &
        (df_daily[COL_VOLUME] > 30)
    ].copy()

    df_put[COL_STRIKE] = pd.to_numeric(df_put[COL_STRIKE], errors='coerce')
    df_put[COL_PRICE] = pd.to_numeric(df_put[COL_PRICE], errors='coerce')
    df_put = df_put.dropna(subset=[COL_STRIKE, COL_PRICE])

    if df_put.empty:
        print(f"   [{date}] 無符合條件 (Put, Volume>30) 的資料。")
        continue

    print(f"🚀 [{date}] 處理 {target_zip_name} (賣權)，共 {len(df_put)} 筆符合條件，運算中...")

    unique_pairs = df_put[[COL_STRIKE, COL_PRICE]].drop_duplicates()
    unique_pairs['Implied_Volatility'] = unique_pairs.apply(
        lambda x: implied_volatility_bisection_put(
            S=S0, K=x[COL_STRIKE], T=T, r=Rf, market_price=x[COL_PRICE]
        ), axis=1
    )
    df_put = df_put.merge(unique_pairs, on=[COL_STRIKE, COL_PRICE], how='left')

    df_put['Date'] = date
    df_put['File'] = target_csv_name
    df_put['S0'] = S0
    df_put['Contract'] = target_contract
    df_put['ContractExpiryDate'] = contract_expiry
    df_put['Rf'] = Rf
    df_put['買賣權別'] = 'P' # 🌟 強制加上賣權標記

    final_results.append(df_put)

# ==========================================
# 7. 彙整結果、輸出 (🌟 修改存檔路徑，直接存到 Colab 本機目錄)
# ==========================================
if final_results:
    df_final = pd.concat(final_results, ignore_index=True)

    # 🌟 確保輸出欄位包含買賣權別 🌟
    keep_columns = [
        'Date', 'File', 'S0', 'Contract', '買賣權別', 'ContractExpiryDate', 'Rf',
        COL_STRIKE, COL_PRICE, 'Implied_Volatility'
    ]
    df_final = df_final[keep_columns]

    # 🚀 終極防呆解法：直接存在 /content/ 底下，保證絕對存得進去！
    output_path = '/content/Processed_Options_PUT_IV_2022_Final.csv'
    df_final.to_csv(output_path, index=False, encoding='utf-8-sig')

    print("\n" + "⭐"*30)
    print("📊 2022 每日【賣權】隱含波動率 (IV) 敘述統計：")
    print("⭐"*30)

    iv_stats = df_final.groupby('Date')['Implied_Volatility'].describe()
    display(iv_stats)

    # 🚀 統計檔也一樣存在 /content/ 底下
    stats_output_path = '/content/IV_Daily_Statistics_PUT_2022_Final.csv'
    iv_stats.to_csv(stats_output_path, encoding='utf-8-sig')

    print(f"\n✅ 賣權運算與存檔完美完成！")
    print(f"👉 請點擊左側檔案總管的「重新整理」圖示 📁")
    print(f"📂 逐筆明細資料已產生：{output_path}")
    print(f"📈 每日敘述統計已產生：{stats_output_path}")
    print(f"趕快把它們下載下來吧！")
else:
    print("\n❌ 運算失敗：未能成功解析內部資料。")

🔄 正在掛載 Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

📊 正在讀取 Excel 索引檔...
✅ 成功讀取索引檔，共 246 筆設定。



/tmp/ipykernel_19430/2689680269.py:68: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, encoding=encoding, on_bad_lines='skip')


🚀 [2022-12-30] 處理 OptionsDaily_2022_12_30.zip (賣權)，共 163 筆符合條件，運算中...


/tmp/ipykernel_19430/2689680269.py:68: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, encoding=encoding, on_bad_lines='skip')


🚀 [2022-12-29] 處理 OptionsDaily_2022_12_29.zip (賣權)，共 112 筆符合條件，運算中...
⚠️ [2022-12-28] 找不到壓縮檔：OptionsDaily_2022_12_28.zip，已略過。


...

🚀 [2022-01-04] 處理 OptionsDaily_2022_01_04.zip (賣權)，共 277 筆符合條件，運算中...


/tmp/ipykernel_19430/2689680269.py:68: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, encoding=encoding, on_bad_lines='skip')


🚀 [2022-01-03] 處理 OptionsDaily_2022_01_03.zip (賣權)，共 210 筆符合條件，運算中...

⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐
📊 2022 每日【賣權】隱含波動率 (IV) 敘述統計：
⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐


,count,mean,std,min,25%,50%,75%,max
Date,,,,,,,,
2022-01-03,210.0,0.191359,0.034499,0.129666,0.164246,0.179031,0.214467,0.331296
2022-01-04,277.0,0.220377,0.290434,0.149005,0.177253,0.189274,0.231695,5.000000
2022-01-05,188.0,0.188376,0.035932,0.140917,0.155791,0.168672,0.227231,0.254381
2022-01-06,297.0,0.218848,0.044234,0.090837,0.181142,0.219914,0.254442,0.307334
2022-01-07,210.0,0.210091,0.053015,0.067150,0.174938,0.209415,0.250466,0.404505
...,...,...,...,...,...,...,...,...
2022-12-23,68.0,0.221587,0.027460,0.193378,0.207757,0.216302,0.226623,0.353366
2022-12-26,42.0,0.212008,0.016664,0.183290,0.199889,0.209619,0.226467,0.263149
2022-12-27,79.0,0.200598,0.023332,0.153589,0.177338,0.206563,0.219787,0.243046



✅ 賣權運算與存檔完美完成！
👉 請點擊左側檔案總管的「重新整理」圖示 📁
📂 逐筆明細資料已產生：/content/Processed_Options_PUT_IV_2022_Final.csv
📈 每日敘述統計已產生：/content/IV_Daily_Statistics_PUT_2022_Final.csv
趕快把它們下載下來吧！


In [ ]:
import pandas as pd

# 1. 讀取你已經算好的賣權明細檔
file_path = '/content/Processed_Options_PUT_IV_2022_Final.csv'
df = pd.read_csv(file_path)

# 2. 檢查一下有多少筆資料撞到 5.0 天花板
outliers = df[df['Implied_Volatility'] >= 4.99]
print(f"🚨 抓到了！共有 {len(outliers)} 筆資料的 IV 異常飆高（接近或等於 5.0）。")

# 3. 剔除異常值（我們只保留 IV 小於 4.99 的正常合理資料）
df_clean = df[df['Implied_Volatility'] < 4.99].copy()

# 4. 重新計算每日敘述統計
iv_stats_clean = df_clean.groupby('Date')['Implied_Volatility'].describe()

# 5. 輸出清洗後的乾淨檔案
clean_data_path = '/content/Processed_Options_PUT_IV_2022_Final_Clean.csv'
clean_stats_path = '/content/IV_Daily_Statistics_PUT_2022_Final_Clean.csv'

df_clean.to_csv(clean_data_path, index=False, encoding='utf-8-sig')
iv_stats_clean.to_csv(clean_stats_path, encoding='utf-8-sig')

print("\n" + "⭐"*30)
print("📊 清洗後的 2022 每日【賣權】隱含波動率 (IV) 敘述統計：")
print("⭐"*30)
display(iv_stats_clean)

print(f"\n✅ 異常值已成功剔除！")
print(f"📂 乾淨的明細資料已存為：{clean_data_path}")
print(f"📈 乾淨的敘述統計已存為：{clean_stats_path}")

🚨 抓到了！共有 14 筆資料的 IV 異常飆高（接近或等於 5.0）。

⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐
📊 清洗後的 2022 每日【賣權】隱含波動率 (IV) 敘述統計：
⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐


,count,mean,std,min,25%,50%,75%,max
Date,,,,,,,,
2022-01-03,210.0,0.191359,0.034499,0.129666,0.164246,0.179031,0.214467,0.331296
2022-01-04,276.0,0.203059,0.035858,0.149005,0.177253,0.188754,0.231695,0.356315
2022-01-05,188.0,0.188376,0.035932,0.140917,0.155791,0.168672,0.227231,0.254381
2022-01-06,297.0,0.218848,0.044234,0.090837,0.181142,0.219914,0.254442,0.307334
2022-01-07,210.0,0.210091,0.053015,0.067150,0.174938,0.209415,0.250466,0.404505
...,...,...,...,...,...,...,...,...
2022-12-23,68.0,0.221587,0.027460,0.193378,0.207757,0.216302,0.226623,0.353366
2022-12-26,42.0,0.212008,0.016664,0.183290,0.199889,0.209619,0.226467,0.263149
2022-12-27,79.0,0.200598,0.023332,0.153589,0.177338,0.206563,0.219787,0.243046



✅ 異常值已成功剔除！
📂 乾淨的明細資料已存為：/content/Processed_Options_PUT_IV_2022_Final_Clean.csv
📈 乾淨的敘述統計已存為：/content/IV_Daily_Statistics_PUT_2022_Final_Clean.csv


In [ ]:
import pandas as pd

# ==========================================
# 1. 設定檔案路徑 (請確認檔名與你實際下載的相符)
# ==========================================
# 買權的每日統計檔 (你一開始跑出來的)
call_stats_path = '/content/drive/MyDrive/金融資料探勘/IVData_411336018_2022.csv'
# 賣權的每日統計檔 (清洗掉 5.0 極端值之後的版本)
put_stats_path = '/content/drive/MyDrive/金融資料探勘/IV_Daily_Statistics_PUT_2022_Final_Clean.csv'

print("讀取資料中...")
try:
    df_call = pd.read_csv(call_stats_path)
    df_put = pd.read_csv(put_stats_path)
except FileNotFoundError as e:
    print(f"❌ 找不到檔案，請確認檔案是否已上傳至 Colab: {e}")
    # 若有錯誤，請提前結束
    raise

# ==========================================
# 2. 擷取需要的欄位並重新命名
# ==========================================
# 我們只需要 Date, count (筆數), mean (平均數), std (標準差)
df_call_sub = df_call[['Date', 'count', 'mean', 'std']].copy()
df_call_sub.columns = ['Date', 'Call_Count', 'Call_IV_Mean', 'Call_IV_Std']

df_put_sub = df_put[['Date', 'count', 'mean', 'std']].copy()
df_put_sub.columns = ['Date', 'Put_Count', 'Put_IV_Mean', 'Put_IV_Std']

# ==========================================
# 3. 將買權與賣權資料根據「日期」合併
# ==========================================
# 使用 inner join 確保只保留兩邊都有資料的日期
df_master = pd.merge(df_call_sub, df_put_sub, on='Date', how='inner')

# ==========================================
# 4. 計算每日 Put/Call Ratio (基於有效交易筆數)
# ==========================================
# PCR = 賣權筆數 / 買權筆數
df_master['PCR (Trade Count)'] = df_master['Put_Count'] / df_master['Call_Count']

# 為了版面美觀，我們把數字四捨五入一下 (平均數/標準差留4位，PCR留4位)
df_master['Call_IV_Mean'] = df_master['Call_IV_Mean'].round(4)
df_master['Call_IV_Std'] = df_master['Call_IV_Std'].round(4)
df_master['Put_IV_Mean'] = df_master['Put_IV_Mean'].round(4)
df_master['Put_IV_Std'] = df_master['Put_IV_Std'].round(4)
df_master['PCR (Trade Count)'] = df_master['PCR (Trade Count)'].round(4)

# 調整欄位順序讓對比更清晰
final_cols = [
    'Date',
    'Call_IV_Mean', 'Put_IV_Mean',
    'Call_IV_Std', 'Put_IV_Std',
    'Call_Count', 'Put_Count', 'PCR (Trade Count)'
]
df_master = df_master[final_cols]

# ==========================================
# 5. 輸出最終研究大表
# ==========================================
output_master_path = '/content/Master_Research_Table_2022.csv'
df_master.to_csv(output_master_path, index=False, encoding='utf-8-sig')

print("\n" + "⭐"*30)
print("📊 2022 買賣權 IV 特徵與 PCR 總表預覽：")
print("⭐"*30)
display(df_master.head(10)) # 顯示前10筆預覽

print(f"\n✅ 完美整合！")
print(f"📂 最終研究總表已存為：{output_master_path}")
print("👉 請點擊左側檔案總管下載，這張表可以直接貼進你的論文或報告中了！")

讀取資料中...

⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐
📊 2022 買賣權 IV 特徵與 PCR 總表預覽：
⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐⭐


,Date,Call_IV_Mean,Put_IV_Mean,Call_IV_Std,Put_IV_Std,Call_Count,Put_Count,PCR (Trade Count)
0,2022-01-03,0.1185,0.1914,0.0138,0.0345,206.0,210.0,1.0194
1,2022-01-04,0.1017,0.2031,0.0258,0.0359,425.0,276.0,0.6494
2,2022-01-05,0.1212,0.1884,0.0305,0.0359,183.0,188.0,1.0273
3,2022-01-06,0.1159,0.2188,0.0262,0.0442,261.0,297.0,1.1379
4,2022-01-07,0.1483,0.2101,0.0240,0.0530,269.0,210.0,0.7807
5,2022-01-10,0.1271,0.2434,0.0182,0.0484,174.0,143.0,0.8218
6,2022-01-11,0.1180,0.2200,0.0226,0.0300,250.0,261.0,1.0440
7,2022-01-12,0.1175,0.2138,0.0132,0.0581,582.0,627.0,1.0773
8,2022-01-19,0.1259,0.2592,0.0176,0.0570,109.0,76.0,0.6972
9,2022-01-20,0.1247,0.2410,0.0243,0.0459,244.0,210.0,0.8607



✅ 完美整合！
📂 最終研究總表已存為：/content/Master_Research_Table_2022.csv
👉 請點擊左側檔案總管下載，這張表可以直接貼進你的論文或報告中了！


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import os
import zipfile
import io
import warnings

warnings.filterwarnings('ignore')

# --- 1. 路徑設定 (雲端硬碟掛載) ---
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

base_path = "/content/drive/MyDrive/金融資料探勘"
index_file_path = os.path.join(base_path, "Index_411336018_2022.xlsx")
# 依照您指定的主壓縮檔路徑
master_zip_path = os.path.join(base_path, "Option_2022.zip")

# 輸出四個目標檔案
out_f1 = os.path.join(base_path, "Options_IV_Summary_2022.csv")
out_f2 = os.path.join(base_path, "Call_IV_Details_2022.csv")
out_f3 = os.path.join(base_path, "Put_IV_Details_2022.csv")
out_f4 = os.path.join(base_path, "PCR_Report_2022.csv")

# --- 2. Black-Scholes 模型與二分法 ---
def black_scholes(S, K, T, r, sigma, option_type='C'):
    if sigma <= 0 or S <= 0 or K <= 0 or T <= 0: return 0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == 'C':
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def find_iv(market_price, S, K, T, r, option_type='C'):
    if market_price <= 0: return 0
    intrinsic = max(0, S - K) if option_type == 'C' else max(0, K * np.exp(-r * T) - S)
    if market_price <= intrinsic: return 0.0001
    low, high = 0.0001, 3.0
    for _ in range(35):
        mid = (low + high) / 2
        price = black_scholes(S, K, T, r, mid, option_type)
        if price < market_price: low = mid
        else: high = mid
    return mid

# --- 3. 遞迴穿透雙層壓縮檔工具 ---
def read_target_csv_from_zip(zip_obj, target_date_str):
    """
    處理雙重壓縮：在 Option_2022.zip 中尋找 OptionsDaily_2022_01_04.zip，
    並直接在記憶體中解開讀取 CSV。
    """
    all_names = zip_obj.namelist()

    # 步驟 1：看這層有沒有直接暴露的 CSV
    csv_matches = [n for n in all_names if n.endswith('.csv') and target_date_str in n.replace("_", "")]
    if csv_matches:
        with zip_obj.open(csv_matches[0]) as f:
            content = f.read()
            for enc in ['big5', 'utf-8-sig', 'gbk']:
                try:
                    df = pd.read_csv(io.BytesIO(content), encoding=enc, low_memory=False)
                    if '商品代號' not in "".join(df.columns.astype(str)):
                         df = pd.read_csv(io.BytesIO(content), encoding=enc, skiprows=1, low_memory=False)
                    return df
                except:
                    continue

    # 步驟 2：找同日期的 ZIP (例如 OptionsDaily_2022_01_04.zip)
    zip_matches = [n for n in all_names if n.endswith('.zip') and target_date_str in n.replace("_", "")]
    if zip_matches:
        with zip_obj.open(zip_matches[0]) as f:
            inner_zip = zipfile.ZipFile(io.BytesIO(f.read()))
            return read_target_csv_from_zip(inner_zip, target_date_str)

    # 步驟 3：防呆備案，外層用月份包裝 (例如 2022_01.zip)
    month_str = target_date_str[:6]
    month_zip_matches = [n for n in all_names if n.endswith('.zip') and month_str in n.replace("_", "") and "Daily" not in n]
    if month_zip_matches:
        with zip_obj.open(month_zip_matches[0]) as f:
            inner_zip = zipfile.ZipFile(io.BytesIO(f.read()))
            return read_target_csv_from_zip(inner_zip, target_date_str)

    return None

# --- 4. 核心處理程序 ---
def process_2022_all_in_one():
    print(f" 正在讀取 2022 索引檔與主壓縮檔...")
    if not os.path.exists(index_file_path):
        print(f"❌ 找不到索引檔：{index_file_path}")
        return
    if not os.path.exists(master_zip_path):
        print(f"❌ 找不到壓縮檔：{master_zip_path}")
        return

    print("✅ 檔案路徑確認無誤！")

    # 讀取 Excel 索引檔
    df_index = pd.read_excel(index_file_path)
    df_index.columns = df_index.columns.str.strip()
    if 'Date' in df_index.columns[0]: df_index.rename(columns={df_index.columns[0]: 'Date'}, inplace=True)

    all_details = []
    print(f" 🚀 開始自動穿透 Option_2022.zip 處理數據 (支援雙重解壓縮)...")

    with zipfile.ZipFile(master_zip_path, 'r') as master_zip:
        for _, row in df_index.iterrows():
            date_str = str(row['Date']).split(' ')[0].replace("-", "").replace("/", "").strip()

            s0, rf = float(str(row['S0']).replace(',', '')), float(row['Rf'])
            t_years = float(row['Maturity']) / 365.0
            target_contract = str(row['Contract']).strip()
            expiry_date = str(row['ContractExpiryDate']).strip()

            print(f"[{date_str}] 處理中...", end=" ")

            raw_df = read_target_csv_from_zip(master_zip, date_str)

            if raw_df is not None:
                raw_df.columns = raw_df.columns.astype(str).str.strip()
                if '商品代號' in raw_df.columns: raw_df['商品代號'] = raw_df['商品代號'].str.strip()
                if '買賣權別' in raw_df.columns: raw_df['買賣權別'] = raw_df['買賣權別'].astype(str).str.strip()

                for col in ['履約價格', '成交價格', '成交數量(B or S)']:
                    if col in raw_df.columns:
                        raw_df[col] = pd.to_numeric(raw_df[col], errors='coerce')

                mask = (raw_df['商品代號'] == 'TXO') & \
                       (raw_df['到期月份(週別)'].astype(str).str.contains(target_contract)) & \
                       (raw_df['成交數量(B or S)'] > 30) & \
                       (raw_df['買賣權別'].isin(['C', 'P']))

                df_filtered = raw_df[mask].copy()

                if not df_filtered.empty:
                    for idx, opt_row in df_filtered.iterrows():
                        iv = find_iv(opt_row['成交價格'], s0, opt_row['履約價格'], t_years, rf, opt_row['買賣權別'])
                        all_details.append({
                            'Date': row['Date'],
                            'File': f"o{date_str}.csv",
                            'S0': s0,
                            'Contract': target_contract,
                            '買賣權別': opt_row['買賣權別'],
                            'ContractExpiryDate': expiry_date,
                            'Rf': rf,
                            '履約價格': opt_row['履約價格'],
                            '成交價格': opt_row['成交價格'],
                            'Implied_Volatility': iv
                        })
                    print(f"✅ 完成 ({len(df_filtered)} 筆交易)")
                else:
                    print("⚠️ 無符合條件交易")
            else:
                print("❌ 找不到檔案")

    if not all_details:
        print("\n 失敗：2022 年度未提取到任何資料。")
        return

    print("\n 正在產出 2022 分析報表...")
    df_main = pd.DataFrame(all_details)

    # 1. 總摘要表
    df_main.groupby('Date')['Implied_Volatility'].describe().to_csv(out_f1, encoding='utf-8-sig')

    # 2 & 3. 買賣權明細
    cols = ['Date', 'File', 'S0', 'Contract', '買賣權別', 'ContractExpiryDate', 'Rf', '履約價格', '成交價格', 'Implied_Volatility']
    for cp, path in [('C', out_f2), ('P', out_f3)]:
        df_main[df_main['買賣權別'] == cp][cols].to_csv(path, index=False, encoding='utf-8-sig')

    # 4. PCR 綜合報表 (僅保留 Trade Count，徹底刪除 Volume)
    stats_iv = df_main.dropna(subset=['Implied_Volatility']).groupby(['Date', '買賣權別'])['Implied_Volatility'].agg(['mean', 'std', 'count']).unstack()

    pcr = pd.DataFrame(index=stats_iv.index)
    pcr['Call_IV_Mean'], pcr['Put_IV_Mean'] = stats_iv[('mean', 'C')], stats_iv[('mean', 'P')]
    pcr['Call_IV_Std'], pcr['Put_IV_Std'] = stats_iv[('std', 'C')], stats_iv[('std', 'P')]
    pcr['Call_Count'], pcr['Put_Count'] = stats_iv[('count', 'C')].fillna(0), stats_iv[('count', 'P')].fillna(0)

    # 僅計算筆數 PCR
    pcr['PCR (Trade Count)'] = np.where(pcr['Call_Count'] == 0, np.nan, pcr['Put_Count'] / pcr['Call_Count'])

    pcr.to_csv(out_f4, encoding='utf-8-sig')
    print(f"✨ 2022 年度處理大功告成！所有報表已儲存。")

process_2022_all_in_one()

 正在讀取 2022 索引檔與主壓縮檔...
✅ 檔案路徑確認無誤！
 🚀 開始自動穿透 Option_2022.zip 處理數據 (支援雙重解壓縮)...
[20221230] 處理中... ✅ 完成 (3021 筆交易)
[20221229] 處理中... ✅ 完成 (2226 筆交易)
[20221228] 處理中... ✅ 完成 (1172 筆交易)
[20221227] 處理中... ✅ 完成 (306 筆交易)
[20221226] 處理中... ✅ 完成 (225 筆交易)
[20221223] 處理中... ✅ 完成 (284 筆交易)
[20221222] 處理中... ✅ 完成 (624 筆交易)
[20221221] 處理中... ✅ 完成 (213 筆交易)
[20221220] 處理中... ✅ 完成 (6907 筆交易)
[20221219] 處理中... ✅ 完成 (4361 筆交易)
[20221216] 處理中... ✅ 完成 (5183 筆交易)
[20221215] 處理中... ✅ 完成 (3264 筆交易)
[20221214] 處理中... ✅ 完成 (10405 筆交易)
[20221213] 處理中... ✅ 完成 (3102 筆交易)
[20221212] 處理中... ✅ 完成 (3462 筆交易)
[20221209] 處理中... ✅ 完成 (2640 筆交易)
[20221208] 處理中... ✅ 完成 (2661 筆交易)
[20221207] 處理中... ✅ 完成 (10250 筆交易)
[20221206] 處理中... ✅ 完成 (6048 筆交易)
[20221205] 處理中... ✅ 完成 (4382 筆交易)
[20221202] 處理中... ✅ 完成 (2598 筆交易)
[20221201] 處理中... ✅ 完成 (3904 筆交易)
[20221130] 處理中... ✅ 完成 (1075 筆交易)
[20221129] 處理中... ✅ 完成 (692 筆交易)
[20221128] 處理中... ✅ 完成 (520 筆交易)
[20221125] 處理中... ✅ 完成 (182 筆交易)
[20221124] 處理中... ✅ 完成 (541 筆交易)
[20221123]